# SDS 02 — Stream Sampling, Filtering, and Reservoir Sampling

**Purpose:** reusable exam/server reference notebook.

This notebook covers three problems that sound similar but require different tools:

1. **Reproducible fixed-fraction entity sampling** → deterministic hash sampling.
2. **Approximate membership with no false negatives** → Bloom filter.
3. **Exactly `k` uniformly selected items from an unknown-length stream** → reservoir sampling.

The notebook deliberately avoids third-party hashing packages such as `mmh3`. Spark examples use Spark SQL hash functions; standalone Bloom-filter code uses only the Python standard library.

> Exam habit: before coding, write one sentence that names the problem type and why the chosen algorithm matches the required guarantee.

## 0. Recognition table

| Prompt clue | First thought | Key guarantee |
|---|---|---|
| “same devices every run”, “1% of devices”, “do not store sampled IDs” | Hash sampling | deterministic entity decision; approximately fraction `p` |
| “known set”, “no false negatives”, “small false-positive rate” | Bloom filter | negative is definite; positive is only possible membership |
| “exactly k”, “equal probability”, “among all items seen so far” | Reservoir sampling | after `n` arrivals, every item has probability `k/n` |
| “independently sample 1% of events” | Bernoulli/event sampling | each event independently kept with probability `p` |

Do not choose an algorithm just because the prompt contains the word **random**.

## 1. Pure-Python deterministic hash helper
Useful for sanity tests outside Spark. This is not Python's built-in `hash()`, because `hash()` is not stable enough for persistent cross-process reproducibility.

In [ ]:
import hashlib

def stable_u64(value, salt=42):
    """Stable unsigned 64-bit hash using Python stdlib only."""
    payload = f"{salt}|{value}".encode("utf-8")
    return int.from_bytes(hashlib.blake2b(payload, digest_size=8).digest(), "big")

def hash_sample_entity(entity_id, fraction, salt=42, buckets=10_000):
    """Deterministic entity sampling. fraction must be in [0,1]."""
    if not (0 <= fraction <= 1):
        raise ValueError("fraction must be between 0 and 1")
    threshold = round(fraction * buckets)
    return stable_u64(entity_id, salt) % buckets < threshold

# Example: the decision for an entity is stable.
for x in ["dev-A", "dev-B", "dev-C"]:
    print(x, hash_sample_entity(x, 0.01, salt=42))

### Why hash the entity ID?
If the requirement says **sample devices**, hash `tracker_id`. If you hash `(tracker_id, timestamp)` or the whole event, the same device can be selected for one event and rejected for another. That is event sampling, not entity sampling.

## 2. Spark-native reproducible entity sampling

In [ ]:
# Run this cell on the exam/server where PySpark is available.
from pyspark.sql import functions as F

def deterministic_entity_sample(df, id_col, numerator=1, denominator=100, salt=42):
    """
    Keep approximately numerator/denominator of distinct entities.
    All rows with the same id_col receive the same decision.
    """
    if not (0 <= numerator <= denominator):
        raise ValueError("Require 0 <= numerator <= denominator")

    bucket = F.pmod(
        F.xxhash64(F.col(id_col), F.lit(int(salt))),
        F.lit(int(denominator))
    )
    return df.filter(bucket < F.lit(int(numerator)))

# Example for 1% of trackers:
# sampled = deterministic_entity_sample(events, "tracker_id", 1, 100, salt=42)

### Critical exam check
For 1%, the correct bucket rule is conceptually:

```text
hash(id) mod 100 < 1
```

The supplied exam's flawed code used `hash % 1`, which is always zero and therefore sampled everything. Always test a modulo expression mentally with a few integers before trusting it.

## 3. Sanity-check the observed sample fraction in Spark

In [ ]:
def observed_entity_fraction(original_df, sampled_df, id_col):
    total = original_df.select(id_col).distinct().count()
    kept  = sampled_df.select(id_col).distinct().count()
    return {"total_entities": total, "sampled_entities": kept,
            "observed_fraction": kept / total if total else None}

# Example:
# observed_entity_fraction(events, sampled, "tracker_id")

The observed proportion will generally be **close to**, not exactly equal to, the requested percentage. A deterministic hash threshold provides a probability/partition rule, not an exact cardinality constraint.

## 4. Bloom-filter parameter formulas

In [ ]:
import math

def bloom_parameters(n, target_fpr, round_bits_to=None):
    """Return practical integer m and k that satisfy the requested approximate FPR."""
    if n <= 0:
        raise ValueError("n must be positive")
    if not (0 < target_fpr < 1):
        raise ValueError("target_fpr must be between 0 and 1")

    m_real = -n * math.log(target_fpr) / (math.log(2) ** 2)

    if round_bits_to:
        m = int(math.ceil(m_real / round_bits_to) * round_bits_to)
        step = int(round_bits_to)
    else:
        m = int(math.ceil(m_real))
        step = 1

    # Rounding m and k can move the achieved FPR slightly above the target.
    # Search upward until a nearby integer k actually satisfies the requirement.
    while True:
        k_real = (m / n) * math.log(2)
        candidates = sorted({
            max(1, int(math.floor(k_real))),
            max(1, int(round(k_real))),
            max(1, int(math.ceil(k_real))),
        })
        scored = []
        for k in candidates:
            achieved = (1 - math.exp(-k * n / m)) ** k
            scored.append((achieved, k))
        achieved, k = min(scored)
        if achieved <= target_fpr:
            break
        m += step

    return {
        "n": n,
        "target_fpr": target_fpr,
        "m_real": m_real,
        "m_bits": m,
        "m_bytes": math.ceil(m / 8),
        "k_real": (m / n) * math.log(2),
        "k_hashes": k,
        "approx_fpr": achieved,
    }

bloom_parameters(500, 0.02, round_bits_to=1024)

For the observed exam values `n=500` and target false-positive rate `p=0.02`, a convenient choice is:

- `m = 4096` bits = 512 bytes
- `k = 6` hashes
- approximate FPR ≈ 1.96%

The exam lesson is not merely “use a Bloom filter.” **Sizing is part of the answer.**

## 5. Standalone Bloom filter — no third-party package

In [ ]:
class BloomFilter:
    """Small educational Bloom filter using only Python stdlib."""
    def __init__(self, m_bits, k_hashes, salt=42):
        self.m = int(m_bits)
        self.k = int(k_hashes)
        self.salt = int(salt)
        self.bits = bytearray((self.m + 7) // 8)

    def _positions(self, item):
        # Double hashing: derive two 64-bit values, then h_i = h1 + i*h2.
        payload = f"{self.salt}|{item}".encode("utf-8")
        digest = hashlib.blake2b(payload, digest_size=16).digest()
        h1 = int.from_bytes(digest[:8], "big")
        h2 = int.from_bytes(digest[8:], "big") or 1
        for i in range(self.k):
            yield (h1 + i * h2) % self.m

    def _set_bit(self, pos):
        self.bits[pos // 8] |= (1 << (pos % 8))

    def _get_bit(self, pos):
        return (self.bits[pos // 8] >> (pos % 8)) & 1

    def add(self, item):
        for pos in self._positions(item):
            self._set_bit(pos)

    def might_contain(self, item):
        return all(self._get_bit(pos) for pos in self._positions(item))

# Exam-like example
bf = BloomFilter(m_bits=4096, k_hashes=6, salt=42)
approved = [f"device-{i}" for i in range(500)]
for d in approved:
    bf.add(d)

assert all(bf.might_contain(d) for d in approved)  # no false negatives here
print("bytes used:", len(bf.bits))
print("known approved:", bf.might_contain("device-10"))
print("unknown device:", bf.might_contain("device-999999"))

### Bloom-filter logic to say out loud

- If **any** required bit is zero → definitely absent.
- If **all** required bits are one → possibly present.
- Inserted items are not rejected as long as bits are never incorrectly cleared.
- A non-member can be accepted because its positions may already have been set by other members.

A standard Bloom filter is therefore one-sided: **false positives yes, false negatives no**.

## 6. Spark hash-position pattern for Bloom-style filtering

In [ ]:
# Produces k deterministic hash positions using only Spark SQL functions.
def add_bloom_positions(df, id_col, m_bits=4096, k_hashes=6):
    out = df
    for seed in range(k_hashes):
        out = out.withColumn(
            f"h{seed}",
            F.pmod(
                F.xxhash64(F.col(id_col), F.lit(seed)),
                F.lit(m_bits)
            )
        )
    return out

# positions = add_bloom_positions(approved_devices, "device_id", 4096, 6)

For only a few hundred approved IDs, the Bloom state is intentionally tiny. It is reasonable to construct that small state once and broadcast it while keeping the enormous event stream distributed. Do **not** collect the event stream to the driver merely to run membership tests locally.

## 7. Reservoir sampling — Algorithm R

In [ ]:
import random

def reservoir_sample(iterable, k, seed=42):
    """Uniform sample of size k from an iterable of unknown length."""
    if k < 0:
        raise ValueError("k must be nonnegative")
    rng = random.Random(seed)
    reservoir = []

    for i, item in enumerate(iterable):
        if i < k:
            reservoir.append(item)
        else:
            # i is zero-based, so there are i+1 items seen including current item.
            j = rng.randint(0, i)
            if j < k:
                reservoir[j] = item
    return reservoir

print(reservoir_sample(range(100), 5, seed=42))

### The two reservoir formulas to remember

For the `i`-th arriving item under **1-based** counting (`i > k`):

```text
P(new item is accepted) = k / i
```

After `n` items have been processed:

```text
P(any particular item is in the reservoir) = k / n
```

The second statement is the guarantee you should be prepared to justify.

## 8. Reservoir uniformity proof — compact exam version

Assume that after `n` items, every old item is in the reservoir with probability `k/n`. At arrival `n+1`:

1. The new item is accepted with probability `k/(n+1)`.
2. A particular occupied reservoir slot is replaced with probability
   `k/(n+1) × 1/k = 1/(n+1)`.
3. Therefore an old sampled item survives with probability `n/(n+1)`.
4. Its final inclusion probability is
   `(k/n) × (n/(n+1)) = k/(n+1)`.

So the new item and every old item have the same final probability `k/(n+1)`.

## 9. Streaming reservoir state object — handy for simulation

In [ ]:
class ReservoirState:
    def __init__(self, k, seed=42):
        self.k = int(k)
        self.rng = random.Random(seed)
        self.n_seen = 0
        self.items = []

    def update(self, item):
        self.n_seen += 1
        i = self.n_seen  # one-based
        if len(self.items) < self.k:
            self.items.append(item)
        else:
            j = self.rng.randint(1, i)
            if j <= self.k:
                self.items[j - 1] = item
        return self

state = ReservoirState(5, seed=42)
for x in range(100):
    state.update(x)
print(state.items, state.n_seen)

## 10. Empirical sanity check for reservoir uniformity

In [ ]:
def reservoir_inclusion_experiment(n=20, k=5, trials=20_000):
    counts = [0] * n
    for seed in range(trials):
        sample = reservoir_sample(range(n), k, seed=seed)
        for x in sample:
            counts[x] += 1
    probs = [c / trials for c in counts]
    return probs

probs = reservoir_inclusion_experiment(n=20, k=5, trials=5_000)
print("target:", 5/20)
print("min/max empirical:", min(probs), max(probs))

This simulation is not the proof; it is a debugging tool. The proof gives the guarantee. The simulation helps catch coding mistakes such as an off-by-one error in `randint`.

## 11. Parallel-friendly uniform top-k priority sample

In [ ]:
# Useful Spark reference for a finite/batch DataFrame.
# Requires a stable unique event_id.
def deterministic_priority_sample(df, event_id_col, k, salt=2026):
    priority = F.pmod(
        F.xxhash64(F.col(event_id_col), F.lit(int(salt))),
        F.lit(2**31 - 1)
    )
    return df.withColumn("_priority", priority).orderBy("_priority").limit(int(k))

# sample_k = deterministic_priority_sample(events, "event_id", 5000)

This is **not the procedural Reservoir Algorithm R**. It is a parallel-friendly random-priority view: if every item receives an independent uniform continuous priority, the `k` smallest priorities form a uniform size-`k` sample. In an exam, name the distinction instead of silently replacing one algorithm with the other.

## 12. Fixed fraction vs fixed count — quick diagnostic

In [ ]:
def recommend_sampling_method(requirement):
    req = requirement.lower()
    if "no false negative" in req or "membership" in req or "approved" in req:
        return "Bloom filter / approximate membership"
    if "exactly" in req or "fixed size" in req or "among all" in req:
        return "Reservoir sampling (or uniform random-priority top-k)"
    if "reproduc" in req or "same device" in req or "same user" in req:
        return "Deterministic hash sampling of the entity ID"
    return "Could be Bernoulli/event sampling; inspect the guarantee carefully"

for q in [
    "reproducibly sample 1% of devices",
    "exactly 5000 among all items seen so far",
    "approved IDs, no false negatives, 2% false positives",
]:
    print(q, "->", recommend_sampling_method(q))

## 13. Bloom-filter exam calculator

In [ ]:
def bloom_fpr(n, m, k):
    return (1 - math.exp(-k * n / m)) ** k

# Check a proposed answer rather than trusting it.
for m, k in [(2048, 8), (4096, 6)]:
    print(m, k, bloom_fpr(500, m, k))

### Exam habit: verify after rounding
If a prompt specifies a maximum error/FPR, do not stop after deriving real-valued formulas. Pick integer parameters, plug them back into the achieved-error formula, and state explicitly whether the requirement is met.

## 14. Critique template — copy/adapt under time pressure

### Correct aspects
1. **Algorithm choice:** Why the named method matches the guarantee.
2. **State/update logic:** What part of the implementation is structurally right.
3. **Complexity/reproducibility:** What requirement is satisfied.

### Limitations / errors
1. **Arithmetic or probability:** Recompute modulo, FPR, inclusion probability, etc.
2. **Parameters:** Are all required parameters explicitly chosen and justified?
3. **Scalability / exact requirement:** Fixed count vs expected count? Entity vs event? Driver collection? Third-party package?

### Verdict
One sentence: correct method but wrong parameters; correct idea but wrong implementation; wrong problem interpretation; or fully valid with stated assumptions.

## 15. Exam-style mini drills

**A.** Reproducibly keep 0.5% of customer accounts, including all events from selected accounts. Give Spark parameters and code.

**B.** Insert 20,000 IDs into a Bloom filter with FPR ≤ 0.005. Derive `m` and `k`, then verify the rounded design.

**C.** Keep exactly 25,000 readings uniformly from an indefinitely growing stream. State the algorithm, the acceptance probability of item `i`, and the inclusion probability after `n` items.

**D.** Critique: “Use reservoir sampling with `k=1%` so the same 1% of users will always be selected.”

## 16. Solutions to mini drills

In [ ]:
# A: 0.5% = 5 / 1000, or 50 / 10000.
# Example Spark call:
# sampled = deterministic_entity_sample(events, "customer_id", 5, 1000, salt=42)

# B:
print(bloom_parameters(20_000, 0.005))

# C:
# Reservoir size k = 25_000.
# P(item i accepted at arrival) = k/i for i>k.
# P(any item remains after n arrivals) = k/n.

# D:
# Reservoir k is a fixed integer sample size, not a percentage and not stable entity membership.
# For a reproducible fraction of users, hash user_id into deterministic buckets.

## 17. Final memory block

**HASH SAMPLE**  
Fixed fraction of persistent entities. Same entity → same decision. `hash(entity_id) mod B < T`. Memory `O(1)`.

**BLOOM FILTER**  
Approximate membership. False positives possible; false negatives absent under the standard model. Size from `n` and target FPR.

**RESERVOIR**  
Exactly `k` uniformly from unknown-length stream. New item `i` enters with `k/i`; after `n`, every item has inclusion probability `k/n`. Memory `O(k)`.

Next notebooks will cover **distinct counting / HyperLogLog / Count-Min Sketch** and then **moments / DGIM / decaying windows**.